In [27]:
import sys
sys.path.insert(0,"../../../")

import joblib
import pandas as pd
import json
from typing import List, Dict, Tuple, Set
from copy import deepcopy
import ast

from src.utils.data_structs import NodeType, RelationType, Triplet
from collections import defaultdict

In [65]:
BASE_DIR = "../../.." # TO CHANGE
SELECTED_BENCHMARK_NAME = "2wiki" # TO CHANGE
SELECTED_KNOWLEDGEGRAPH_NAME = "qwen257b_080226_v2prompts" # TO CHANGE

QA_DATASETS_DIR = f"{BASE_DIR}/data/qa_datasets"
KNOWLEDGE_GRAPHS_DIR = f"{BASE_DIR}/../pai2_knowledge_graphs"
EXTRACTED_TRIPLES_FILENAME = 'extracted_triplets'
TRIPLES_NODES_IDS_FILENAME = 'partial_triples_nodes_ids.joblib'
MATCHED_TNIDS_TO_QUESTIONS_FILENAME = 'matched_tnids_to_questions.joblib'

BENCHMARK_PATH = {
    'nq': {
        'qa': f"{QA_DATASETS_DIR}/natural_questions_train/qa_pairs.csv",
        'documents': f"{QA_DATASETS_DIR}/natural_questions_train/relevant_contexts.csv"
    }, 
    'triviaqa': {
        'qa': f"{QA_DATASETS_DIR}/v2/trivia_qa_rcwikipedia_validation/qa_pairs.csv",
        'documents': f"{QA_DATASETS_DIR}/v2/trivia_qa_rcwikipedia_validation/relevant_contexts.csv"
    }, 
    'hotpotqa': {
        'qa': f"{QA_DATASETS_DIR}/v2/hotpotqa_distractor_validation/qa_pairs.csv",
        'documents': f"{QA_DATASETS_DIR}/v2/hotpotqa_distractor_validation/relevant_contexts.csv"
    }, 
    '2wiki': {
        'qa': f"{QA_DATASETS_DIR}/2wikimultihopqa_dev/qa_pairs.csv",
        'documents': f"{QA_DATASETS_DIR}/2wikimultihopqa_dev/relevant_contexts.csv"
    }, 
    'musique': {
        'qa': f"{QA_DATASETS_DIR}/musique_validation/qa_pairs.csv",
        'documents': f"{QA_DATASETS_DIR}/musique_validation/relevant_contexts.csv"
    }
}

EXTRACTED_TRIPLES_PATH = f"{KNOWLEDGE_GRAPHS_DIR}/{SELECTED_BENCHMARK_NAME}/{SELECTED_KNOWLEDGEGRAPH_NAME}/{EXTRACTED_TRIPLES_FILENAME}"
SAVE_TRIPLES_NODES_IDS_PATH = f"{KNOWLEDGE_GRAPHS_DIR}/{SELECTED_BENCHMARK_NAME}/{SELECTED_KNOWLEDGEGRAPH_NAME}/{TRIPLES_NODES_IDS_FILENAME}"
SAVE_MATCHED_TNIDS_TO_QUESTIONS_PATH = f"{KNOWLEDGE_GRAPHS_DIR}/{SELECTED_BENCHMARK_NAME}/{SELECTED_KNOWLEDGEGRAPH_NAME}/{MATCHED_TNIDS_TO_QUESTIONS_FILENAME}"

In [10]:
EXTRACTED_TRIPLES = joblib.load(EXTRACTED_TRIPLES_PATH)

/home/dzigen/miniconda3/envs/pai/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
print(len(EXTRACTED_TRIPLES))

4596


In [44]:
len(EXTRACTED_TRIPLES[0])

107

In [14]:
BENCHMARK_DOCUMENTS = pd.read_csv(BENCHMARK_PATH[SELECTED_BENCHMARK_NAME]['documents'], sep=',')
BENCHMARK_QA = pd.read_csv(BENCHMARK_PATH[SELECTED_BENCHMARK_NAME]['qa'], sep=',')
print(BENCHMARK_DOCUMENTS.shape)
print(BENCHMARK_QA.shape)

(4596, 4)
(2000, 6)


In [12]:
BENCHMARK_DOCUMENTS.head()

,title,context,cntx_idx,hash
0,Polish-Russian War (film),Polish-Russian War (Wojna polsko-ruska) is a 2...,0,a23632c49331d3e7158a339adb16fdff
1,Xawery Żuławski,Xawery Żuławski (born 22 December 1971 in Wars...,1,03e78a2115c7401abe1985ca88ddc58e
2,Blind Shaft,Blind Shaft is a 2003 film about a pair of bru...,2,5f81eca1a49862947f6076fa910f5593
3,The Mask of Fu Manchu,The Mask of Fu Manchu is a 1932 pre-Code adven...,3,fbfd94b02d71d66981158e9de0e4a675
4,"John V, Prince of Anhalt-Zerbst","John V of Anhalt-Zerbst (Dessau, 4 September 1...",4,e046c19181bb7924533d964b34f72114


In [13]:
BENCHMARK_QA.head()

,question,type,answer,cntx_ids,qa_idx,original_qaidx
0,Who is the mother of the director of film Poli...,compositional,Małgorzata Braunek,"[0, 1]",0,0
1,"Which film came out first, Blind Shaft or The ...",comparison,The Mask Of Fu Manchu,"[2, 3]",1,1
2,"When did John V, Prince Of Anhalt-Zerbst's fat...",compositional,12 June 1516,"[4, 5]",2,2
3,What is the award that the director of film We...,compositional,Myanmar Motion Picture Academy Awards,"[6, 7]",3,3
4,Who is Charles Bretagne Marie De La Trémoille'...,inference,Charles Armand René de La Trémoille,"[8, 9]",4,5


In [69]:
EXTRACTED_TRIPLES_NODES_IDS: Dict = dict()
MATCHED_TNIDS_TO_QUESTIONS: List[Dict] = []

In [70]:
for idx in range(BENCHMARK_DOCUMENTS.shape[0]):
    cur_cntx_idx = BENCHMARK_DOCUMENTS['cntx_idx'][idx]
    cur_triples: List[Triplet] = EXTRACTED_TRIPLES[idx]

    uniques_object_nodes_info: Set[Tuple[str,str]] = set()
    uniques_thesis_nodes_info: Set[Tuple[str,str]] = set()
    unique_triples_ids: Dict[str, Set[str]] = defaultdict(set)
    tripletid_to_thesisid: Dict[str, str] = dict()
    for cur_triplet in cur_triples:
        if cur_triplet.start_node.type == NodeType.object:
            uniques_object_nodes_info.add((cur_triplet.start_node.id, cur_triplet.start_node.name))
        elif cur_triplet.start_node.type == NodeType.hyper:
            uniques_thesis_nodes_info.add((cur_triplet.start_node.id, cur_triplet.start_node.name))
            tripletid_to_thesisid[cur_triplet.id] = cur_triplet.start_node.id
        
        if cur_triplet.end_node.type == NodeType.object:
            uniques_object_nodes_info.add((cur_triplet.end_node.id, cur_triplet.end_node.name))
        elif cur_triplet.end_node.type == NodeType.hyper:
            uniques_thesis_nodes_info.add((cur_triplet.end_node.id, cur_triplet.end_node.name))
            tripletid_to_thesisid[cur_triplet.id] = cur_triplet.end_node.id
                    
        unique_triples_ids[cur_triplet.relation.type.value].add(cur_triplet.id)

    EXTRACTED_TRIPLES_NODES_IDS[int(cur_cntx_idx)] = {
        'unique_object_nodes_info': deepcopy(uniques_object_nodes_info),
        'uniques_thesis_nodes_info': deepcopy(uniques_thesis_nodes_info),
        'unique_triples_ids': deepcopy(dict(unique_triples_ids)),
        'tripletid_to_thesisid': deepcopy(tripletid_to_thesisid)
    }
EXTRACTED_TRIPLES_NODES_IDS = dict(EXTRACTED_TRIPLES_NODES_IDS)

In [71]:
len(EXTRACTED_TRIPLES_NODES_IDS)

4596

In [72]:
EXTRACTED_TRIPLES_NODES_IDS[0]

{'unique_object_nodes_info': {('01be28d032bfcda36f22b45ad3298bec',
   'last film'),
  ('175bcb21510404da1eb0458ac8dd5af3', 'kerwin mathews'),
  ('282f5dc7fac457d1217e4fd19918d158', 'andré hunebelle'),
  ('296cc9a02b167493f6ff885c48ca71ff', 'colour'),
  ('2bb937d2cf231c100f37e8cc7cc5a889', 'da. ma. cinematografica'),
  ('3910983de59d8413765574090521b4ad', 'international co-production'),
  ('3939151107d9f1c791788e164aba2db4', '74th oss 117 novel'),
  ('39dcaf7a053dc372fbc391d4e6b5d693', '1964'),
  ('3f77b7e9a85aa618e0aa5b2f998583e9', 'action scenes'),
  ('5436b6230ef21650b7e92de67d107650', 'first oss 117 film'),
  ('5f3fd601ca27ab1e58fd1286a8df0452', 'jean bruce'),
  ('7d87e4cbf6234bd3d6cbd59706af3603', 'co-production'),
  ('7f16109f1619fd7a733daf5a84c708c1', '1960'),
  ('8149ed381cb0798ec5643f0285d120c5', 'second oss 117 film'),
  ('83a1a1c1da46c00a08618d063a6abf11', 'shot'),
  ('8689391a8b93cd2d55ccf3f436eef4e2', 'thailand'),
  ('95cf9ab0cf87766af408a02c24aa1c11', 'production design'),

In [73]:
for idx in range(BENCHMARK_QA.shape[0]):
    qa_idx = BENCHMARK_QA['qa_idx'][idx]
    relevant_cntx_idxs = ast.literal_eval(BENCHMARK_QA['cntx_ids'][idx])

    uniques_object_nodes_info: Set[Tuple[str,str]] = set()
    uniques_thesis_nodes_info: Set[Tuple[str,str]] = set()
    unique_triples_ids: Dict[str, Set[str]] = defaultdict(set)
    tripletid_to_thesisid: Dict[str, str] = dict()

    for cur_cntx_idx in relevant_cntx_idxs:
        uniques_object_nodes_info.update(EXTRACTED_TRIPLES_NODES_IDS[cur_cntx_idx]['unique_object_nodes_info'])
        uniques_thesis_nodes_info.update(EXTRACTED_TRIPLES_NODES_IDS[cur_cntx_idx]['uniques_thesis_nodes_info'])
        tripletid_to_thesisid.update(EXTRACTED_TRIPLES_NODES_IDS[cur_cntx_idx]['tripletid_to_thesisid'])
        for k, v in EXTRACTED_TRIPLES_NODES_IDS[cur_cntx_idx]['unique_triples_ids'].items():
            unique_triples_ids[k].update(v)


    MATCHED_TNIDS_TO_QUESTIONS.append({
        'qa_idx': int(qa_idx),
        'question': BENCHMARK_QA['question'][idx],
        'cntx_idxs': deepcopy(relevant_cntx_idxs),
        'relevant_unique_object_nodes_info': deepcopy(uniques_object_nodes_info),
        'relevant_unique_thesis_nodes_info': deepcopy(uniques_thesis_nodes_info),
        'relevant_tripletid_to_thesisid': deepcopy(tripletid_to_thesisid),
        'relevant_unique_triples_ids': deepcopy(dict(unique_triples_ids))
    })

In [74]:
print(len(MATCHED_TNIDS_TO_QUESTIONS))

2000


In [75]:
MATCHED_TNIDS_TO_QUESTIONS[0]

{'qa_idx': 0,
 'question': 'Who is the mother of the director of film Polish-Russian War (Film)?',
 'cntx_idxs': [0, 1],
 'relevant_unique_object_nodes_info': {('01be28d032bfcda36f22b45ad3298bec',
   'last film'),
  ('0302cdd79c48c15830805aba44f35d5a', 'mystery science theater 3000'),
  ('175bcb21510404da1eb0458ac8dd5af3', 'kerwin mathews'),
  ('23a58bf9274bedb19375e527a0744fa9', 'with'),
  ('282f5dc7fac457d1217e4fd19918d158', 'andré hunebelle'),
  ('296cc9a02b167493f6ff885c48ca71ff', 'colour'),
  ('2bb937d2cf231c100f37e8cc7cc5a889', 'da. ma. cinematografica'),
  ('2cf594c51ff9e6dcbc78e42313671e43', 'comedy central'),
  ('3910983de59d8413765574090521b4ad', 'international co-production'),
  ('3939151107d9f1c791788e164aba2db4', '74th oss 117 novel'),
  ('39dcaf7a053dc372fbc391d4e6b5d693', '1964'),
  ('3b2eb19d98fa94a5794f3d6949cf6020', 'herschell gordon lewis'),
  ('3f77b7e9a85aa618e0aa5b2f998583e9', 'action scenes'),
  ('4460740cd9ee7f43a4fde3a966f0787d', 'horror'),
  ('5436b6230ef21650

In [77]:
joblib.dump(EXTRACTED_TRIPLES_NODES_IDS, SAVE_TRIPLES_NODES_IDS_PATH)

['../../../../pai2_knowledge_graphs/2wiki/qwen257b_080226_v2prompts/partial_triples_nodes_ids.joblib']

In [78]:
joblib.dump(MATCHED_TNIDS_TO_QUESTIONS, SAVE_MATCHED_TNIDS_TO_QUESTIONS_PATH)

['../../../../pai2_knowledge_graphs/2wiki/qwen257b_080226_v2prompts/matched_tnids_to_questions.joblib']